In [1]:
#IMPORTS AND SETUP
import pandas as pd
import numpy as np
import json
import re
import warnings
from pathlib import Path
from collections import defaultdict
import pickle

# Fuzzy matching
from rapidfuzz import process, fuzz

# Sentence transformers for text embeddings
from sentence_transformers import SentenceTransformer

# Sklearn utilities
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# PyTorch
import torch
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')

print("All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Paths
ML1M_PATH = Path("data/ml-1m")
TMDB_PATH =  Path("data/tmdb_5000")
OUTPUT_PATH = Path("data/working")
OUTPUT_PATH.mkdir(exist_ok=True)



print(f"\nPaths configured:")
print(f"  ML-1M: {ML1M_PATH}")
print(f"  TMDB:  {TMDB_PATH}")
print(f"  Output: {OUTPUT_PATH}")


/opt/conda/lib/python3.12/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2026-05-11 12:16:01.202388: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-11 12:16:02.959243: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778501763.620618     630 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778501763.825927     

All imports successful!
PyTorch version: 2.6.0+cu126
CUDA available: True

Paths configured:
  ML-1M: data/ml-1m
  TMDB:  data/tmdb_5000
  Output: data/working


In [2]:
# LOAD MOVIELENS 1M DATA

print("="*80)
print("LOADING MOVIELENS 1M DATA")
print("="*80)

# Load ratings.dat
print("\n[1/3] Loading ratings.dat...")
ratings_df = pd.read_csv(
    ML1M_PATH / "ratings.dat",
    sep="::",
    engine="python",
    names=["UserID", "MovieID", "Rating", "Timestamp"],
    encoding="latin-1"
)
print(f"  Shape: {ratings_df.shape}")
print(f"  Columns: {list(ratings_df.columns)}")
print(f"  Missing values: {ratings_df.isnull().sum().sum()}")
print(f"  UserID range: {ratings_df['UserID'].min()} - {ratings_df['UserID'].max()}")
print(f"  MovieID range: {ratings_df['MovieID'].min()} - {ratings_df['MovieID'].max()}")
print(f"  Rating range: {ratings_df['Rating'].min()} - {ratings_df['Rating'].max()}")
print(f"  Unique users: {ratings_df['UserID'].nunique()}")
print(f"  Unique movies: {ratings_df['MovieID'].nunique()}")

# Load users.dat
print("\n[2/3] Loading users.dat...")
users_df = pd.read_csv(
    ML1M_PATH / "users.dat",
    sep="::",
    engine="python",
    names=["UserID", "Gender", "Age", "Occupation", "Zip-code"],
    encoding="latin-1"
)
print(f"  Shape: {users_df.shape}")
print(f"  Columns: {list(users_df.columns)}")
print(f"  Missing values: {users_df.isnull().sum().sum()}")
print(f"  Gender distribution:\n{users_df['Gender'].value_counts()}")
print(f"  Age groups: {sorted(users_df['Age'].unique())}")
print(f"  Occupation range: {users_df['Occupation'].min()} - {users_df['Occupation'].max()}")

# Load movies.dat
print("\n[3/3] Loading movies.dat...")
movies_df = pd.read_csv(
    ML1M_PATH / "movies.dat",
    sep="::",
    engine="python",
    names=["MovieID", "Title", "Genres"],
    encoding="latin-1"
)
print(f"  Shape: {movies_df.shape}")
print(f"  Columns: {list(movies_df.columns)}")
print(f"  Missing values: {movies_df.isnull().sum().sum()}")
print(f"  Sample titles:\n{movies_df['Title'].head(3).tolist()}")

# Extract all unique genres
all_genres = set()
for genres_str in movies_df['Genres'].dropna():
    all_genres.update(genres_str.split('|'))
all_genres = sorted(all_genres)
print(f"\n  Unique genres ({len(all_genres)}): {all_genres}")

print("\n MovieLens 1M data loaded successfully!")


LOADING MOVIELENS 1M DATA

[1/3] Loading ratings.dat...
  Shape: (1000209, 4)
  Columns: ['UserID', 'MovieID', 'Rating', 'Timestamp']
  Missing values: 0
  UserID range: 1 - 6040
  MovieID range: 1 - 3952
  Rating range: 1 - 5
  Unique users: 6040
  Unique movies: 3706

[2/3] Loading users.dat...
  Shape: (6040, 5)
  Columns: ['UserID', 'Gender', 'Age', 'Occupation', 'Zip-code']
  Missing values: 0
  Gender distribution:
Gender
M    4331
F    1709
Name: count, dtype: int64
  Age groups: [np.int64(1), np.int64(18), np.int64(25), np.int64(35), np.int64(45), np.int64(50), np.int64(56)]
  Occupation range: 0 - 20

[3/3] Loading movies.dat...
  Shape: (3883, 3)
  Columns: ['MovieID', 'Title', 'Genres']
  Missing values: 0
  Sample titles:
['Toy Story (1995)', 'Jumanji (1995)', 'Grumpier Old Men (1995)']

  Unique genres (18): ['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-F

In [3]:
# LOAD TMDB DATA

print("="*80)
print("LOADING TMDB 5000 DATA")
print("="*80)

# Load TMDB movies
print("\n[1/2] Loading tmdb_5000_movies.csv...")
tmdb_movies = pd.read_csv(TMDB_PATH / "tmdb_5000_movies.csv")
print(f"  Shape: {tmdb_movies.shape}")
print(f"  Columns ({len(tmdb_movies.columns)}): {list(tmdb_movies.columns)}")
print(f"  Missing values:\n{tmdb_movies.isnull().sum()[tmdb_movies.isnull().sum() > 0]}")

# Load TMDB credits
print("\n[2/2] Loading tmdb_5000_credits.csv...")
tmdb_credits = pd.read_csv(TMDB_PATH / "tmdb_5000_credits.csv")
print(f"  Shape: {tmdb_credits.shape}")
print(f"  Columns: {list(tmdb_credits.columns)}")
print(f"  Missing values:\n{tmdb_credits.isnull().sum()[tmdb_credits.isnull().sum() > 0]}")

# Merge TMDB movies and credits on movie_id
print("\n[3/3] Merging TMDB movies and credits...")
tmdb_full = tmdb_movies.merge(
    tmdb_credits[['movie_id', 'cast', 'crew']],
    left_on='id',
    right_on='movie_id',
    how='left'
)
print(f"  Merged shape: {tmdb_full.shape}")

print("\n TMDB data loaded successfully!")


LOADING TMDB 5000 DATA

[1/2] Loading tmdb_5000_movies.csv...
  Shape: (4803, 20)
  Columns (20): ['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']
  Missing values:
homepage        3091
overview           3
release_date       1
runtime            2
tagline          844
dtype: int64

[2/2] Loading tmdb_5000_credits.csv...
  Shape: (4803, 4)
  Columns: ['movie_id', 'title', 'cast', 'crew']
  Missing values:
Series([], dtype: int64)

[3/3] Merging TMDB movies and credits...
  Merged shape: (4803, 23)

 TMDB data loaded successfully!


In [4]:
# PARSE TMDB JSON FIELDS

print("="*80)
print("PARSING TMDB JSON FIELDS")
print("="*80)

def safe_json_parse(json_str):
    """Safely parse JSON string, return empty list if invalid."""
    if pd.isna(json_str):
        return []
    try:
        return json.loads(json_str)
    except:
        return []

# Parse genres
print("\n[1/5] Parsing genres...")
tmdb_full['genres_list'] = tmdb_full['genres'].apply(
    lambda x: [g['name'] for g in safe_json_parse(x)]
)
tmdb_full['genres_str'] = tmdb_full['genres_list'].apply(lambda x: '|'.join(x))
print(f"  Sample: {tmdb_full['genres_str'].iloc[0]}")
print(f"  Non-empty: {(tmdb_full['genres_str'] != '').sum()} / {len(tmdb_full)}")

# Parse keywords
print("\n[2/5] Parsing keywords...")
tmdb_full['keywords_list'] = tmdb_full['keywords'].apply(
    lambda x: [k['name'] for k in safe_json_parse(x)]
)
tmdb_full['keywords_str'] = tmdb_full['keywords_list'].apply(lambda x: ' '.join(x))
print(f"  Sample: {tmdb_full['keywords_str'].iloc[0][:100]}...")
print(f"  Non-empty: {(tmdb_full['keywords_str'] != '').sum()} / {len(tmdb_full)}")

# Parse cast (top 5 actors)
print("\n[3/5] Parsing cast...")
def get_top_cast(cast_json, n=5):
    cast_list = safe_json_parse(cast_json)
    return [c['name'] for c in cast_list[:n]]

tmdb_full['cast_list'] = tmdb_full['cast'].apply(get_top_cast)
tmdb_full['cast_str'] = tmdb_full['cast_list'].apply(lambda x: ', '.join(x))
print(f"  Sample: {tmdb_full['cast_str'].iloc[0]}")
print(f"  Non-empty: {(tmdb_full['cast_str'] != '').sum()} / {len(tmdb_full)}")

# Parse crew (extract director)
print("\n[4/5] Parsing crew (director)...")
def get_director(crew_json):
    crew_list = safe_json_parse(crew_json)
    directors = [c['name'] for c in crew_list if c.get('job') == 'Director']
    return directors[0] if directors else ''

tmdb_full['director'] = tmdb_full['crew'].apply(get_director)
print(f"  Sample: {tmdb_full['director'].iloc[0]}")
print(f"  Non-empty: {(tmdb_full['director'] != '').sum()} / {len(tmdb_full)}")

# Extract release year
print("\n[5/5] Extracting release year...")
tmdb_full['release_year'] = pd.to_datetime(
    tmdb_full['release_date'], errors='coerce'
).dt.year
print(f"  Year range: {tmdb_full['release_year'].min()} - {tmdb_full['release_year'].max()}")
print(f"  Missing: {tmdb_full['release_year'].isnull().sum()}")

# Fill missing values
tmdb_full['overview'] = tmdb_full['overview'].fillna('')
tmdb_full['tagline'] = tmdb_full['tagline'].fillna('')
tmdb_full['runtime'] = tmdb_full['runtime'].fillna(0)
tmdb_full['budget'] = tmdb_full['budget'].fillna(0)
tmdb_full['revenue'] = tmdb_full['revenue'].fillna(0)
tmdb_full['popularity'] = tmdb_full['popularity'].fillna(0)
tmdb_full['vote_average'] = tmdb_full['vote_average'].fillna(0)
tmdb_full['vote_count'] = tmdb_full['vote_count'].fillna(0)

print("\n TMDB JSON fields parsed successfully!")


PARSING TMDB JSON FIELDS

[1/5] Parsing genres...
  Sample: Action|Adventure|Fantasy|Science Fiction
  Non-empty: 4775 / 4803

[2/5] Parsing keywords...
  Sample: culture clash future space war space colony society space travel futuristic romance space alien trib...
  Non-empty: 4391 / 4803

[3/5] Parsing cast...
  Sample: Sam Worthington, Zoe Saldana, Sigourney Weaver, Stephen Lang, Michelle Rodriguez
  Non-empty: 4760 / 4803

[4/5] Parsing crew (director)...
  Sample: James Cameron
  Non-empty: 4773 / 4803

[5/5] Extracting release year...
  Year range: 1916.0 - 2017.0
  Missing: 1

 TMDB JSON fields parsed successfully!


In [5]:
# FUZZY MATCH MOVIELENS WITH TMDB

print("="*80)
print("FUZZY MATCHING MOVIELENS WITH TMDB")
print("="*80)

# Parse MovieLens titles to extract year
print("\n[1/3] Parsing MovieLens titles...")
def parse_ml_title(title):
    """Extract movie name and year from 'Movie Name (YYYY)' format."""
    match = re.search(r'^(.+?)\s*\((\d{4})\)$', title)
    if match:
        return match.group(1).strip(), int(match.group(2))
    return title.strip(), None

movies_df['Title_Clean'] = movies_df['Title'].apply(lambda x: parse_ml_title(x)[0])
movies_df['Year'] = movies_df['Title'].apply(lambda x: parse_ml_title(x)[1])
print(f"  Titles with year: {movies_df['Year'].notna().sum()} / {len(movies_df)}")
print(f"  Sample:\n{movies_df[['Title', 'Title_Clean', 'Year']].head(3)}")

# Prepare TMDB lookup
print("\n[2/3] Preparing TMDB lookup...")
tmdb_lookup = tmdb_full[['title', 'release_year']].copy()
tmdb_lookup['title_lower'] = tmdb_lookup['title'].str.lower().str.strip()
print(f"  TMDB movies: {len(tmdb_lookup)}")

# Fuzzy matching function
def normalize_title(t):
    """Move trailing articles back to front and lowercase."""
    t = t.strip().lower()
    # Handle "Title, The" → "the title"
    for article in [', the', ', a', ', an']:
        if t.endswith(article):
            t = article.strip(', ') + ' ' + t[:-len(article)]
            break
    return t

def _get_best_match(norm_query, candidates_df, scorer, threshold):
    """Helper: run extractOne and return (original_title, score) or (None, 0)."""
    choices = candidates_df['title_lower'].apply(normalize_title).tolist()
    result  = process.extractOne(norm_query, choices, scorer=scorer)
    if result is None or result[1] < threshold:
        return None, 0
    best_norm = result[0]
    cdf = candidates_df.copy()
    cdf['title_norm'] = cdf['title_lower'].apply(normalize_title)
    matched = cdf[cdf['title_norm'] == best_norm]
    if len(matched) == 0:
        return None, 0
    return matched.iloc[0]['title'], result[1]


def fuzzy_match_movie(ml_title, ml_year, tmdb_df, threshold=70):
    norm_query = normalize_title(ml_title)

    # Pass 1: strict year ±1, token_set_ratio ≥ threshold
    if pd.notna(ml_year):
        strict = tmdb_df[
            (tmdb_df['release_year'] >= ml_year - 1) &
            (tmdb_df['release_year'] <= ml_year + 1)
        ]
        if len(strict) > 0:
            t, s = _get_best_match(norm_query, strict, fuzz.token_set_ratio, threshold)
            if t:
                return t, s

    # Pass 2: relaxed year ±3, higher threshold (85)
    if pd.notna(ml_year):
        relaxed = tmdb_df[
            (tmdb_df['release_year'] >= ml_year - 3) &
            (tmdb_df['release_year'] <= ml_year + 3)
        ]
        if len(relaxed) > 0:
            t, s = _get_best_match(norm_query, relaxed, fuzz.token_set_ratio, 85)
            if t:
                return t, s

    # Pass 3: no year filter, token_set_ratio ≥ 90
    t, s = _get_best_match(norm_query, tmdb_df, fuzz.token_set_ratio, 90)
    if t:
        return t, s

    # Pass 4: no year filter, partial_ratio ≥ 95 (last resort)
    t, s = _get_best_match(norm_query, tmdb_df, fuzz.partial_ratio, 95)
    return t, s

# Perform fuzzy matching
print("\n[3/3] Matching MovieLens movies with TMDB...")
matches = []
for idx, row in movies_df.iterrows():
    matched_title, score = fuzzy_match_movie(
        row['Title_Clean'],
        row['Year'],
        tmdb_lookup,
        threshold=70
    )
    matches.append({
        'MovieID': row['MovieID'],
        'ML_Title': row['Title'],
        'TMDB_Title': matched_title,
        'Match_Score': score
    })
    
    if (idx + 1) % 500 == 0:
        print(f"  Processed {idx + 1} / {len(movies_df)} movies...")

matches_df = pd.DataFrame(matches)
print(f"\n  Total matches: {matches_df['TMDB_Title'].notna().sum()} / {len(movies_df)}")
print(f"  Match rate: {100 * matches_df['TMDB_Title'].notna().sum() / len(movies_df):.1f}%")
print(f"  Average match score: {matches_df[matches_df['TMDB_Title'].notna()]['Match_Score'].mean():.1f}")

# Show sample matches
print("\n  Sample matches:")
print(matches_df[matches_df['TMDB_Title'].notna()][['ML_Title', 'TMDB_Title', 'Match_Score']].head(10))

print("\n Fuzzy matching completed!")


FUZZY MATCHING MOVIELENS WITH TMDB

[1/3] Parsing MovieLens titles...
  Titles with year: 3883 / 3883
  Sample:
                     Title       Title_Clean  Year
0         Toy Story (1995)         Toy Story  1995
1           Jumanji (1995)           Jumanji  1995
2  Grumpier Old Men (1995)  Grumpier Old Men  1995

[2/3] Preparing TMDB lookup...
  TMDB movies: 4803

[3/3] Matching MovieLens movies with TMDB...
  Processed 500 / 3883 movies...
  Processed 1000 / 3883 movies...
  Processed 1500 / 3883 movies...
  Processed 2000 / 3883 movies...
  Processed 2500 / 3883 movies...
  Processed 3000 / 3883 movies...
  Processed 3500 / 3883 movies...

  Total matches: 3285 / 3883
  Match rate: 84.6%
  Average match score: 98.6

  Sample matches:
                              ML_Title              TMDB_Title  Match_Score
0                     Toy Story (1995)               Toy Story        100.0
2              Grumpier Old Men (1995)                       O        100.0
3             Waiting to

In [6]:
# MERGE MOVIELENS WITH TMDB


print("="*80)
print("MERGING MOVIELENS WITH TMDB")
print("="*80)

# Merge matches with TMDB data
print("\n[1/2] Merging matched movies with TMDB metadata...")
movies_enriched = movies_df.merge(
    matches_df[['MovieID', 'TMDB_Title', 'Match_Score']],
    on='MovieID',
    how='left'
)

# Merge with TMDB full data
movies_enriched = movies_enriched.merge(
    tmdb_full[[
        'title', 'overview', 'genres_str', 'keywords_str', 'cast_str', 'director',
        'budget', 'revenue', 'runtime', 'popularity', 'vote_average', 'vote_count'
    ]],
    left_on='TMDB_Title',
    right_on='title',
    how='left'
)

print(f"  Enriched movies shape: {movies_enriched.shape}")
print(f"  Movies with TMDB data: {movies_enriched['overview'].notna().sum()} / {len(movies_enriched)}")

# Rename columns for clarity
movies_enriched = movies_enriched.rename(columns={
    'Title': 'Title_Original',
    'Genres': 'Genres_ML',
    'genres_str': 'Genres_TMDB',
    'keywords_str': 'Keywords',
    'cast_str': 'Cast',
    'director': 'Director'
})

# Fill missing TMDB fields with defaults
print("\n[2/2] Filling missing values...")
movies_enriched['overview'] = movies_enriched['overview'].fillna('')
movies_enriched['Genres_TMDB'] = movies_enriched['Genres_TMDB'].fillna('')
movies_enriched['Keywords'] = movies_enriched['Keywords'].fillna('')
movies_enriched['Cast'] = movies_enriched['Cast'].fillna('')
movies_enriched['Director'] = movies_enriched['Director'].fillna('')
movies_enriched['budget'] = movies_enriched['budget'].fillna(0)
movies_enriched['revenue'] = movies_enriched['revenue'].fillna(0)
movies_enriched['runtime'] = movies_enriched['runtime'].fillna(0)
movies_enriched['popularity'] = movies_enriched['popularity'].fillna(0)
movies_enriched['vote_average'] = movies_enriched['vote_average'].fillna(0)
movies_enriched['vote_count'] = movies_enriched['vote_count'].fillna(0)

# Select final columns
movies_enriched = movies_enriched[[
    'MovieID', 'Title_Original', 'Title_Clean', 'Year', 'Genres_ML', 'Genres_TMDB',
    'overview', 'Keywords', 'Cast', 'Director',
    'budget', 'revenue', 'runtime', 'popularity', 'vote_average', 'vote_count'
]]

print(f"\n  Final enriched movies shape: {movies_enriched.shape}")
print(f"  Columns: {list(movies_enriched.columns)}")
print(f"\n  Sample enriched movie:")
sample = movies_enriched[movies_enriched['overview'] != ''].iloc[0]
print(f"    Title: {sample['Title_Original']}")
print(f"    Genres (ML): {sample['Genres_ML']}")
print(f"    Genres (TMDB): {sample['Genres_TMDB']}")
print(f"    Overview: {sample['overview'][:150]}...")
print(f"    Cast: {sample['Cast']}")
print(f"    Director: {sample['Director']}")

print("\n MovieLens and TMDB merged successfully!")


MERGING MOVIELENS WITH TMDB

[1/2] Merging matched movies with TMDB metadata...
  Enriched movies shape: (3884, 19)
  Movies with TMDB data: 3286 / 3884

[2/2] Filling missing values...

  Final enriched movies shape: (3884, 16)
  Columns: ['MovieID', 'Title_Original', 'Title_Clean', 'Year', 'Genres_ML', 'Genres_TMDB', 'overview', 'Keywords', 'Cast', 'Director', 'budget', 'revenue', 'runtime', 'popularity', 'vote_average', 'vote_count']

  Sample enriched movie:
    Title: Toy Story (1995)
    Genres (ML): Animation|Children's|Comedy
    Genres (TMDB): Animation|Comedy|Family
    Overview: Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's he...
    Cast: Tom Hanks, Tim Allen, Don Rickles, Jim Varney, Wallace Shawn
    Director: John Lasseter

 MovieLens and TMDB merged successfully!


In [7]:
# CREATE USER FEATURE VECTORS


print("="*80)
print("CREATING USER FEATURE VECTORS")
print("="*80)

# Encode Gender: M → 0, F → 1
print("\n[1/3] Encoding Gender...")
users_df['Gender_Encoded'] = users_df['Gender'].map({'M': 0, 'F': 1})
print(f"  Gender mapping: M → 0, F → 1")
print(f"  Distribution:\n{users_df['Gender_Encoded'].value_counts().sort_index()}")

# Encode Age: Map age groups to indices
print("\n[2/3] Encoding Age...")
age_map = {1: 0, 18: 1, 25: 2, 35: 3, 45: 4, 50: 5, 56: 6}
users_df['Age_Encoded'] = users_df['Age'].map(age_map)
print(f"  Age mapping: {age_map}")
print(f"  Distribution:\n{users_df['Age_Encoded'].value_counts().sort_index()}")

# Encode Occupation: Already 0-20, keep as-is
print("\n[3/3] Encoding Occupation...")
users_df['Occupation_Encoded'] = users_df['Occupation']
print(f"  Occupation range: {users_df['Occupation_Encoded'].min()} - {users_df['Occupation_Encoded'].max()}")
print(f"  Distribution:\n{users_df['Occupation_Encoded'].value_counts().sort_index()}")

# Create final user features dataframe
user_features = users_df[[
    'UserID', 'Gender_Encoded', 'Age_Encoded', 'Occupation_Encoded'
]].copy()

print(f"\n  User features shape: {user_features.shape}")
print(f"  Columns: {list(user_features.columns)}")
print(f"\n  Sample user features:")
print(user_features.head())

print("\n User feature vectors created successfully!")


CREATING USER FEATURE VECTORS

[1/3] Encoding Gender...
  Gender mapping: M → 0, F → 1
  Distribution:
Gender_Encoded
0    4331
1    1709
Name: count, dtype: int64

[2/3] Encoding Age...
  Age mapping: {1: 0, 18: 1, 25: 2, 35: 3, 45: 4, 50: 5, 56: 6}
  Distribution:
Age_Encoded
0     222
1    1103
2    2096
3    1193
4     550
5     496
6     380
Name: count, dtype: int64

[3/3] Encoding Occupation...
  Occupation range: 0 - 20
  Distribution:
Occupation_Encoded
0     711
1     528
2     267
3     173
4     759
5     112
6     236
7     679
8      17
9      92
10    195
11    129
12    388
13    142
14    302
15    144
16    241
17    502
18     70
19     72
20    281
Name: count, dtype: int64

  User features shape: (6040, 4)
  Columns: ['UserID', 'Gender_Encoded', 'Age_Encoded', 'Occupation_Encoded']

  Sample user features:
   UserID  Gender_Encoded  Age_Encoded  Occupation_Encoded
0       1               1            0                  10
1       2               0            6     

In [8]:
# CREATE MOVIE GENRE VECTORS


print("="*80)
print("CREATING MOVIE GENRE VECTORS")
print("="*80)

# Use MovieLens genres (18 unique genres)
print("\n[1/2] Creating multi-hot genre encoding...")
print(f"  Genres: {all_genres}")

# Create multi-hot encoding for each movie
def create_genre_vector(genres_str, all_genres):
    """Convert pipe-separated genres to multi-hot vector."""
    if pd.isna(genres_str) or genres_str == '':
        return np.zeros(len(all_genres))
    
    movie_genres = set(genres_str.split('|'))
    vector = np.array([1 if g in movie_genres else 0 for g in all_genres])
    return vector

movies_enriched['Genre_Vector'] = movies_enriched['Genres_ML'].apply(
    lambda x: create_genre_vector(x, all_genres)
)

print(f"  Genre vector dimension: {len(all_genres)}")
print(f"\n  Sample genre vectors:")
for idx in range(3):
    row = movies_enriched.iloc[idx]
    print(f"    {row['Title_Original']}: {row['Genres_ML']}")
    print(f"      Vector: {row['Genre_Vector']}")

print("\n[2/2] Verifying genre vectors...")
genre_vector_lengths = movies_enriched['Genre_Vector'].apply(len)
print(f"  All vectors have length {len(all_genres)}: {(genre_vector_lengths == len(all_genres)).all()}")
print(f"  Movies with at least one genre: {(movies_enriched['Genre_Vector'].apply(sum) > 0).sum()} / {len(movies_enriched)}")

print("\n Movie genre vectors created successfully!")


CREATING MOVIE GENRE VECTORS

[1/2] Creating multi-hot genre encoding...
  Genres: ['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
  Genre vector dimension: 18

  Sample genre vectors:
    Toy Story (1995): Animation|Children's|Comedy
      Vector: [0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
    Jumanji (1995): Adventure|Children's|Fantasy
      Vector: [0 1 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0]
    Grumpier Old Men (1995): Comedy|Romance
      Vector: [0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0]

[2/2] Verifying genre vectors...
  All vectors have length 18: True
  Movies with at least one genre: 3884 / 3884

 Movie genre vectors created successfully!


In [9]:
# CREATE TEXT EMBEDDINGS WITH SENTENCE-BERT


print("="*80)
print("CREATING TEXT EMBEDDINGS WITH SENTENCE-BERT")
print("="*80)

# Load Sentence-BERT model
print("\n[1/4] Loading Sentence-BERT model (all-MiniLM-L6-v2)...")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"  Model loaded: {sbert_model}")
print(f"  Embedding dimension: {sbert_model.get_sentence_embedding_dimension()}")

# Prepare text for embedding
print("\n[2/4] Preparing text for embedding...")

# Overview embeddings (use title as fallback if overview is empty)
movies_enriched['Text_For_Embedding'] = movies_enriched.apply(
    lambda row: row['overview'] if row['overview'] != '' else row['Title_Original'],
    axis=1
)
print(f"  Using overview: {(movies_enriched['overview'] != '').sum()} / {len(movies_enriched)}")
print(f"  Using title fallback: {(movies_enriched['overview'] == '').sum()} / {len(movies_enriched)}")

# Keywords text (already concatenated)
movies_enriched['Keywords_Text'] = movies_enriched['Keywords'].fillna('')

# Encode overview
print("\n[3/4] Encoding overview text...")
overview_texts = movies_enriched['Text_For_Embedding'].tolist()
overview_embeddings = sbert_model.encode(
    overview_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(f"  Overview embeddings shape: {overview_embeddings.shape}")

# Encode keywords
print("\n[4/4] Encoding keywords text...")
keyword_texts = movies_enriched['Keywords_Text'].tolist()
keyword_embeddings = sbert_model.encode(
    keyword_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(f"  Keyword embeddings shape: {keyword_embeddings.shape}")

# Add embeddings to dataframe
movies_enriched['Overview_Embedding'] = list(overview_embeddings)
movies_enriched['Keyword_Embedding'] = list(keyword_embeddings)

print(f"\n  Sample embedding (first 10 dims):")
print(f"    {movies_enriched['Overview_Embedding'].iloc[0][:10]}")

print("\n✓ Text embeddings created successfully!")


CREATING TEXT EMBEDDINGS WITH SENTENCE-BERT

[1/4] Loading Sentence-BERT model (all-MiniLM-L6-v2)...
  Model loaded: SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)
  Embedding dimension: 384

[2/4] Preparing text for embedding...
  Using overview: 3286 / 3884
  Using title fallback: 598 / 3884

[3/4] Encoding overview text...


Batches:   0%|          | 0/122 [00:00<?, ?it/s]

  Overview embeddings shape: (3884, 384)

[4/4] Encoding keywords text...


Batches:   0%|          | 0/122 [00:00<?, ?it/s]

  Keyword embeddings shape: (3884, 384)

  Sample embedding (first 10 dims):
    [ 0.06343899  0.00102684  0.09321018 -0.01494174 -0.00640027  0.01528889
  0.12308541 -0.03044558 -0.03718008  0.02157226]

✓ Text embeddings created successfully!


In [10]:
# CREATE COMPLETE MOVIE FEATURE VECTORS


print("="*80)
print("CREATING COMPLETE MOVIE FEATURE VECTORS")
print("="*80)

# Normalize numerical features
print("\n[1/2] Normalizing numerical features...")
from sklearn.preprocessing import MinMaxScaler

numerical_features = ['budget', 'revenue', 'runtime', 'popularity', 'vote_average', 'vote_count']

# Fit ONLY on rows that have TMDB data (have a real overview)
# Unmatched rows have 0 for all numerical fields — fitting on them
# would make 0 the min and compress all real values into a tiny range
has_tmdb = movies_enriched['overview'] != ''
scaler   = MinMaxScaler()
scaler.fit(movies_enriched.loc[has_tmdb, numerical_features])

# Transform all rows — unmatched rows (all zeros) will map to ~0.0
movies_enriched[numerical_features] = scaler.transform(
    movies_enriched[numerical_features]
)

print(f"  Scaler fit on {has_tmdb.sum()} TMDB-matched movies")
print(f"  Sample normalized values (matched movie):")
print(movies_enriched.loc[has_tmdb, numerical_features].head(3))
print(f"  Sample normalized values (unmatched movie, should be near 0):")
print(movies_enriched.loc[~has_tmdb, numerical_features].head(3))

# Combine all features into single vector
print("\n[2/2] Combining all features...")

def create_movie_feature_vector(row):
    """
    Combine all movie features into single vector:
    - Genre vector (18 dims)
    - Overview embedding (384 dims)
    - Keyword embedding (384 dims)
    - Numerical features (6 dims)
    Total: 792 dims
    """
    genre_vec = row['Genre_Vector']
    overview_emb = row['Overview_Embedding']
    keyword_emb = row['Keyword_Embedding']
    numerical_vec = row[numerical_features].values
    
    combined = np.concatenate([
        genre_vec,
        overview_emb,
        keyword_emb,
        numerical_vec
    ])
    
    return combined

movies_enriched['Feature_Vector'] = movies_enriched.apply(
    create_movie_feature_vector,
    axis=1
)

# Verify dimensions
feature_dim = len(movies_enriched['Feature_Vector'].iloc[0])
print(f"\n  Feature vector dimension: {feature_dim}")
print(f"    - Genre vector: 18")
print(f"    - Overview embedding: 384")
print(f"    - Keyword embedding: 384")
print(f"    - Numerical features: 6")
print(f"    - Total: {18 + 384 + 384 + 6} = {feature_dim}")

print(f"\n  Sample feature vector (first 20 dims):")
print(f"    {movies_enriched['Feature_Vector'].iloc[0][:20]}")

# Create final movie features dataframe
movie_features = movies_enriched[[
    'MovieID', 'Title_Original', 'Feature_Vector'
]].copy()

print(f"\n  Movie features shape: {movie_features.shape}")
print(f"  All movies have feature vectors: {movie_features['Feature_Vector'].notna().all()}")

print("\n Complete movie feature vectors created successfully!")


CREATING COMPLETE MOVIE FEATURE VECTORS

[1/2] Normalizing numerical features...
  Scaler fit on 3286 TMDB-matched movies
  Sample normalized values (matched movie):
     budget   revenue   runtime  popularity  vote_average  vote_count
0  0.078947  0.202465  0.334711    0.501780          0.77    0.559758
2  0.000000  0.008681  0.392562    0.028248          0.58    0.007968
3  0.000000  0.008681  0.392562    0.028248          0.58    0.007968
  Sample normalized values (unmatched movie, should be near 0):
   budget  revenue  runtime  popularity  vote_average  vote_count
1     0.0      0.0      0.0   -0.000008           0.0         0.0
6     0.0      0.0      0.0   -0.000008           0.0         0.0
8     0.0      0.0      0.0   -0.000008           0.0         0.0

[2/2] Combining all features...

  Feature vector dimension: 792
    - Genre vector: 18
    - Overview embedding: 384
    - Keyword embedding: 384
    - Numerical features: 6
    - Total: 792 = 792

  Sample feature vector (f

In [11]:
# CREATE ID MAPPINGS


print("="*80)
print("CREATING ID MAPPINGS")
print("="*80)

# User ID mapping: 1-6040 → 0-6039
print("\n[1/2] Creating user ID mapping...")
unique_users = sorted(ratings_df['UserID'].unique())
user_id_to_idx = {uid: idx for idx, uid in enumerate(unique_users)}
idx_to_user_id = {idx: uid for uid, idx in user_id_to_idx.items()}

print(f"  Total users: {len(user_id_to_idx)}")
print(f"  UserID range: {min(unique_users)} - {max(unique_users)}")
print(f"  Index range: 0 - {len(user_id_to_idx) - 1}")
print(f"  Sample mapping: UserID 1 → Index {user_id_to_idx[1]}")
print(f"  Sample mapping: UserID 6040 → Index {user_id_to_idx[6040]}")

# Movie ID mapping: Non-contiguous IDs → 0-3705
print("\n[2/2] Creating movie ID mapping...")
# Only map movies that appear in ratings (3706 movies)
unique_movies = sorted(ratings_df['MovieID'].unique())
movie_id_to_idx = {mid: idx for idx, mid in enumerate(unique_movies)}
idx_to_movie_id = {idx: mid for mid, idx in movie_id_to_idx.items()}

print(f"  Total movies: {len(movie_id_to_idx)}")
print(f"  MovieID range: {min(unique_movies)} - {max(unique_movies)}")
print(f"  Index range: 0 - {len(movie_id_to_idx) - 1}")
print(f"  Sample mapping: MovieID 1 → Index {movie_id_to_idx[1]}")
print(f"  Sample mapping: MovieID {max(unique_movies)} → Index {movie_id_to_idx[max(unique_movies)]}")

# Verify all ratings can be mapped
print("\n  Verification:")
unmapped_users = ratings_df[~ratings_df['UserID'].isin(user_id_to_idx.keys())]
unmapped_movies = ratings_df[~ratings_df['MovieID'].isin(movie_id_to_idx.keys())]
print(f"    Unmapped users in ratings: {len(unmapped_users)}")
print(f"    Unmapped movies in ratings: {len(unmapped_movies)}")

print("\n ID mappings created successfully!")


CREATING ID MAPPINGS

[1/2] Creating user ID mapping...
  Total users: 6040
  UserID range: 1 - 6040
  Index range: 0 - 6039
  Sample mapping: UserID 1 → Index 0
  Sample mapping: UserID 6040 → Index 6039

[2/2] Creating movie ID mapping...
  Total movies: 3706
  MovieID range: 1 - 3952
  Index range: 0 - 3705
  Sample mapping: MovieID 1 → Index 0
  Sample mapping: MovieID 3952 → Index 3705

  Verification:
    Unmapped users in ratings: 0
    Unmapped movies in ratings: 0

 ID mappings created successfully!


In [12]:
# Train/Test Split (temporal)

print("="*80)
print("TRAIN / TEST SPLIT")
print("="*80)

# Sort all ratings by timestamp (chronological order)
ratings_sorted = ratings_df.sort_values(['UserID', 'Timestamp']).reset_index(drop=True)

# For each user: last 20% of ratings → test, rest → train
print("\n[1/2] Splitting per-user by time...")
train_rows = []
test_rows  = []

for user_id, group in ratings_sorted.groupby('UserID'):
    group = group.sort_values('Timestamp')
    n = len(group)
    split_idx = max(1, int(n * 0.8))   # at least 1 train sample
    train_rows.append(group.iloc[:split_idx])
    test_rows.append(group.iloc[split_idx:])

train_data = pd.concat(train_rows).reset_index(drop=True)
test_data  = pd.concat(test_rows).reset_index(drop=True)

print(f"  Total ratings : {len(ratings_df):,}")
print(f"  Train ratings : {len(train_data):,}  ({100*len(train_data)/len(ratings_df):.1f}%)")
print(f"  Test  ratings : {len(test_data):,}  ({100*len(test_data)/len(ratings_df):.1f}%)")
print(f"  Train users   : {train_data['UserID'].nunique()}")
print(f"  Test  users   : {test_data['UserID'].nunique()}")

# Verify no temporal leakage: every test timestamp > last train timestamp per user
print("\n[2/2] Verifying temporal integrity...")
leakage_count = 0
for user_id, test_group in test_data.groupby('UserID'):
    train_group = train_data[train_data['UserID'] == user_id]
    if len(train_group) == 0:
        continue
    max_train_ts = train_group['Timestamp'].max()
    min_test_ts  = test_group['Timestamp'].min()
    if min_test_ts < max_train_ts:
        leakage_count += 1

print(f"  Users with temporal leakage: {leakage_count}")
print(f"  {'WARNING: leakage detected!' if leakage_count > 0 else 'No leakage — split is clean.'}")

print("\n Train/test split complete!")


TRAIN / TEST SPLIT

[1/2] Splitting per-user by time...
  Total ratings : 1,000,209
  Train ratings : 797,758  (79.8%)
  Test  ratings : 202,451  (20.2%)
  Train users   : 6040
  Test  users   : 6040

[2/2] Verifying temporal integrity...
  Users with temporal leakage: 0
  No leakage — split is clean.

 Train/test split complete!


In [13]:
# Build user movie history from TRAIN data only

print("="*80)
print("BUILDING USER MOVIE HISTORY")
print("="*80)

from collections import defaultdict

# Build ordered history per user (sorted by timestamp, train only)
# Each entry: (MovieID, Rating, Timestamp)
user_movie_history = defaultdict(list)

for _, row in train_data.sort_values('Timestamp').iterrows():
    user_movie_history[row['UserID']].append({
        'movie_id' : int(row['MovieID']),
        'rating'   : float(row['Rating']),
        'timestamp': int(row['Timestamp'])
    })

print(f"  Users with history : {len(user_movie_history)}")

# Stats
history_lengths = [len(v) for v in user_movie_history.values()]
print(f"  Min history length : {min(history_lengths)}")
print(f"  Max history length : {max(history_lengths)}")
print(f"  Mean history length: {np.mean(history_lengths):.1f}")
print(f"  Median             : {np.median(history_lengths):.1f}")

# Sample
sample_uid = list(user_movie_history.keys())[0]
print(f"\n  Sample user {sample_uid} (first 3 entries):")
for entry in user_movie_history[sample_uid][:3]:
    print(f"    {entry}")

print("\n User movie history built!")


BUILDING USER MOVIE HISTORY
  Users with history : 6040
  Min history length : 16
  Max history length : 1851
  Mean history length: 132.1
  Median             : 76.0

  Sample user 6040 (first 3 entries):
    {'movie_id': 858, 'rating': 4.0, 'timestamp': 956703932}
    {'movie_id': 593, 'rating': 5.0, 'timestamp': 956703954}
    {'movie_id': 2384, 'rating': 4.0, 'timestamp': 956703954}

 User movie history built!


In [14]:
print(user_features.columns.tolist())
print(user_features.head(2))

['UserID', 'Gender_Encoded', 'Age_Encoded', 'Occupation_Encoded']
   UserID  Gender_Encoded  Age_Encoded  Occupation_Encoded
0       1               1            0                  10
1       2               0            6                  16


In [15]:
# Create training sequences (sequence_length = 15)

print("="*80)
print("CREATING TRAINING SEQUENCES")
print("="*80)

SEQUENCE_LENGTH = 15   # number of context movies per sample

# Index user_features by UserID for fast lookup
user_feat_indexed = user_features.set_index('UserID')

# Build movie feature lookup: MovieID → feature vector
movie_feat_lookup = dict(zip(
    movie_features['MovieID'],
    movie_features['Feature_Vector']
))

MOVIE_FEAT_DIM = len(movie_features['Feature_Vector'].iloc[0])
print(f"  Sequence length  : {SEQUENCE_LENGTH}")
print(f"  Movie feature dim: {MOVIE_FEAT_DIM}")

# ── Sequence construction ──────────────────────────────────────────────────
# For each user, slide a window over their chronological history:
#   context = previous SEQUENCE_LENGTH rated movies
#   target  = next movie
# Each sample stores:
#   seq_movie_ids   : (15,)   int   — context movie IDs
#   seq_ratings     : (15,)   float — context ratings (1-5)
#   seq_positions   : (15,)   int   — position indices 0-14
#   target_movie_id : int
#   target_rating   : float
#   user_id         : int

sequences = []
skipped_users = 0

for user_id, history in user_movie_history.items():
    # Need at least SEQUENCE_LENGTH + 1 ratings to form one sample
    if len(history) < SEQUENCE_LENGTH + 1:
        skipped_users += 1
        continue

    # Only keep movies that exist in our movie_id_to_idx mapping
    valid_history = [h for h in history if h['movie_id'] in movie_id_to_idx]

    if len(valid_history) < SEQUENCE_LENGTH + 1:
        skipped_users += 1
        continue

# Sliding window

    for i in range(SEQUENCE_LENGTH, len(valid_history)):
        context = valid_history[i - SEQUENCE_LENGTH : i]
        target  = valid_history[i]

        sequences.append({
            'user_id'          : user_id,
            'seq_movie_ids'    : [c['movie_id'] for c in context],
            'seq_ratings'      : [c['rating']   for c in context],
            'seq_positions'    : list(range(SEQUENCE_LENGTH)),
            'target_movie_id'  : target['movie_id'],
            'target_rating'    : target['rating'],
            'target_timestamp' : target['timestamp'],
        })

print(f"\n  Users skipped (too few ratings): {skipped_users}")
print(f"  Total sequences created        : {len(sequences):,}")

# Quick sanity check
sample_seq = sequences[0]
print(f"\n  Sample sequence:")
print(f"    user_id         : {sample_seq['user_id']}")
print(f"    seq_movie_ids   : {sample_seq['seq_movie_ids']}")
print(f"    seq_ratings     : {sample_seq['seq_ratings']}")
print(f"    seq_positions   : {sample_seq['seq_positions']}")
print(f"    target_movie_id : {sample_seq['target_movie_id']}")
print(f"    target_rating   : {sample_seq['target_rating']}")

print("\nTraining sequences created!")


CREATING TRAINING SEQUENCES
  Sequence length  : 15
  Movie feature dim: 792

  Users skipped (too few ratings): 0
  Total sequences created        : 707,158

  Sample sequence:
    user_id         : 6040
    seq_movie_ids   : [858, 593, 2384, 1961, 2019, 213, 1419, 573, 3111, 3505, 1734, 919, 912, 2503, 527]
    seq_ratings     : [4.0, 5.0, 4.0, 4.0, 5.0, 5.0, 3.0, 4.0, 5.0, 4.0, 2.0, 5.0, 5.0, 5.0, 5.0]
    seq_positions   : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
    target_movie_id : 318
    target_rating   : 4.0

Training sequences created!


In [16]:
print("="*80)
print("DEFINING PYTORCH DATASET")
print("="*80)
print("  NOTE: Dataset class is defined in Model Training notebook.")
print("  Sequences are saved as plain dicts with raw 1-5 ratings.")
print("  Model Training converts ratings to long ints for embedding lookup.")
print()
print("  Dataset class ready!")

DEFINING PYTORCH DATASET
  NOTE: Dataset class is defined in Model Training notebook.
  Sequences are saved as plain dicts with raw 1-5 ratings.
  Model Training converts ratings to long ints for embedding lookup.

  Dataset class ready!


In [17]:
# Instantiate datasets and DataLoaders
BATCH_SIZE = 256
print("="*80)
print("CREATING DATASETS AND DATALOADERS")
print("="*80)

# ── Build test sequences with a running context window ─────────────────────
# For each user, the context grows as we predict each test rating in order.
# After predicting test movie N, it enters the context for predicting N+1.
# This mirrors real inference behaviour.
print("\n[1/3] Building test sequences (dynamic running context)...")
test_sequences = []
skipped_test   = 0

# Sort test_data so we process each user's test ratings in timestamp order
test_data_sorted = test_data.sort_values(['UserID', 'Timestamp'])

for uid, user_test in test_data_sorted.groupby('UserID'):
    # Seed the running context with the user's full training history
    train_hist  = user_movie_history.get(uid, [])
    valid_hist  = [h for h in train_hist if h['movie_id'] in movie_id_to_idx]
    running_ctx = list(valid_hist)   # mutable copy — grows as we process test rows

    for _, row in user_test.iterrows():
        tmid = int(row['MovieID'])

        if tmid not in movie_id_to_idx:
            skipped_test += 1
            # Still add to context so later predictions aren't broken
            running_ctx.append({
                'movie_id' : tmid,
                'rating'   : float(row['Rating']),
                'timestamp': int(row['Timestamp'])
            })
            continue

        if len(running_ctx) < SEQUENCE_LENGTH:
            skipped_test += 1
            running_ctx.append({
                'movie_id' : tmid,
                'rating'   : float(row['Rating']),
                'timestamp': int(row['Timestamp'])
            })
            continue

        context = running_ctx[-SEQUENCE_LENGTH:]
        test_sequences.append({
            'user_id'        : uid,
            'seq_movie_ids'  : [c['movie_id'] for c in context],
            'seq_ratings'    : [c['rating']   for c in context],
            'seq_positions'  : list(range(SEQUENCE_LENGTH)),
            'target_movie_id': tmid,
            'target_rating'  : float(row['Rating']),
        })

        # Append rated test movie to running context for the next prediction
        running_ctx.append({
            'movie_id' : tmid,
            'rating'   : float(row['Rating']),
            'timestamp': int(row['Timestamp'])
        })

print(f"  Test sequences created : {len(test_sequences):,}")
print(f"  Test samples skipped   : {skipped_test:,}")

# ── Train/val split from training sequences ────────────────────────────────
print("\n[2/3] Splitting train sequences into train/val (90/10)...")

# Temporal split: sort by target_timestamp so val is always in the future
sequences.sort(key=lambda x: x['target_timestamp'])
val_size   = int(0.1 * len(sequences))
train_size = len(sequences) - val_size
train_seq  = sequences[:train_size]
val_seq    = sequences[train_size:]
print(f"  Temporal split — train ends at ts={train_seq[-1]['target_timestamp']}, "
      f"val starts at ts={val_seq[0]['target_timestamp']}")

print(f"  Train sequences: {len(train_seq):,}")
print(f"  Val   sequences: {len(val_seq):,}")

print(f"  Train sequences: {len(train_seq):,}")
print(f"  Val   sequences: {len(val_seq):,}")
print(f"  Test  sequences: {len(test_sequences):,}")

# ── Dataset instantiation commented out because Dataset class is defined elsewhere ──
print("\n[3/3] Dataset and DataLoader instantiation skipped in Data Prep.")

print(f"\n  Batch size       : {BATCH_SIZE}")
print(f"  Train batches    : N/A")
print(f"  Val   batches    : N/A")
print(f"  Test  batches    : N/A")

print("\n DataPrep phase complete. Sequences ready for Model Training notebook.")


CREATING DATASETS AND DATALOADERS

[1/3] Building test sequences (dynamic running context)...
  Test sequences created : 202,451
  Test samples skipped   : 0

[2/3] Splitting train sequences into train/val (90/10)...
  Temporal split — train ends at ts=976864563, val starts at ts=976864588
  Train sequences: 636,443
  Val   sequences: 70,715
  Train sequences: 636,443
  Val   sequences: 70,715
  Test  sequences: 202,451

[3/3] Dataset and DataLoader instantiation skipped in Data Prep.

  Batch size       : 256
  Train batches    : N/A
  Val   batches    : N/A
  Test  batches    : N/A

 DataPrep phase complete. Sequences ready for Model Training notebook.


In [18]:
# Save all Phase 1 artifacts to disk

print("="*80)
print("SAVING PHASE 1 ARTIFACTS")
print("="*80)

artifacts = {
    'user_features'      : user_features,
    'movie_features'     : movie_features,
    'movies_enriched'    : movies_enriched,
    'train_data'         : train_data,
    'test_data'          : test_data,
    'user_id_to_idx'     : user_id_to_idx,
    'idx_to_user_id'     : idx_to_user_id,
    'movie_id_to_idx'    : movie_id_to_idx,
    'idx_to_movie_id'    : idx_to_movie_id,
    'user_movie_history' : dict(user_movie_history),
    'train_sequences'    : train_seq,
    'val_sequences'      : val_seq,
    'test_sequences'     : test_sequences,
    'all_genres'         : all_genres,
    'SEQUENCE_LENGTH'    : SEQUENCE_LENGTH,
    'MOVIE_FEAT_DIM'     : MOVIE_FEAT_DIM,
    'NUM_USERS'          : len(user_id_to_idx),
    'NUM_MOVIES'         : len(movie_id_to_idx),
}

artifacts['BATCH_SIZE']   = BATCH_SIZE
artifacts['NUM_USERS']    = len(user_id_to_idx)   
artifacts['NUM_MOVIES']   = len(movie_id_to_idx) 
artifacts['numerical_scaler'] = scaler

for name, obj in artifacts.items():
    path = OUTPUT_PATH / f"{name}.pkl"
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    size_mb = path.stat().st_size / 1e6
    print(f"  Saved {name:30s} → {path}  ({size_mb:.1f} MB)")

print("\n" + "="*80)
print("PHASE 1 COMPLETE — SUMMARY")
print("="*80)
print(f"  Users            : {len(user_id_to_idx):,}")
print(f"  Movies           : {len(movie_id_to_idx):,}")
print(f"  Movie feat dim   : {MOVIE_FEAT_DIM}")
print(f"  Sequence length  : {SEQUENCE_LENGTH}")
print(f"  Train sequences  : {len(train_seq):,}")
print(f"  Val   sequences  : {len(val_seq):,}")
print(f"  Test  sequences  : {len(test_sequences):,}")

print(f"\n  All artifacts saved to: {OUTPUT_PATH}")
print("\n Phase 1 complete")


SAVING PHASE 1 ARTIFACTS
  Saved user_features                  → data/working/user_features.pkl  (0.2 MB)
  Saved movie_features                 → data/working/movie_features.pkl  (27.5 MB)
  Saved movies_enriched                → data/working/movies_enriched.pkl  (41.6 MB)
  Saved train_data                     → data/working/train_data.pkl  (25.5 MB)
  Saved test_data                      → data/working/test_data.pkl  (6.5 MB)
  Saved user_id_to_idx                 → data/working/user_id_to_idx.pkl  (0.1 MB)
  Saved idx_to_user_id                 → data/working/idx_to_user_id.pkl  (0.1 MB)
  Saved movie_id_to_idx                → data/working/movie_id_to_idx.pkl  (0.1 MB)
  Saved idx_to_movie_id                → data/working/idx_to_movie_id.pkl  (0.1 MB)
  Saved user_movie_history             → data/working/user_movie_history.pkl  (21.6 MB)
  Saved train_sequences                → data/working/train_sequences.pkl  (166.2 MB)
  Saved val_sequences                  → data/working/val_

In [19]:
# ── PHASE 1 SANITY CHECKS ─────────────────────────────────────────────────
NUM_USERS = len(user_id_to_idx)
NUM_MOVIES = len(movie_id_to_idx)
print("="*80)
print("PHASE 1 SANITY CHECKS")
print("="*80)

# Check sequence structure
sample = train_seq[0]
assert 'user_id'         in sample, "missing user_id"
assert 'seq_movie_ids'   in sample, "missing seq_movie_ids"
assert 'seq_ratings'     in sample, "missing seq_ratings"
assert 'seq_positions'   in sample, "missing seq_positions"
assert 'target_movie_id' in sample, "missing target_movie_id"
assert 'target_rating'   in sample, "missing target_rating"
print("  [OK] Sequence dict has all required keys")

# Check sequence lengths
assert len(sample['seq_movie_ids']) == SEQUENCE_LENGTH,  "wrong seq_movie_ids length"
assert len(sample['seq_ratings'])   == SEQUENCE_LENGTH,  "wrong seq_ratings length"
assert len(sample['seq_positions']) == SEQUENCE_LENGTH,  "wrong seq_positions length"
print(f"  [OK] All sequences have length {SEQUENCE_LENGTH}")

# Check ratings are raw 1-5 (NOT normalized — Model Training uses them as embedding indices)
all_ratings = [r for s in train_seq[:1000] for r in s['seq_ratings']]
assert all(1 <= r <= 5 for r in all_ratings), "ratings must be raw 1-5 integers"
assert all(1 <= s['target_rating'] <= 5 for s in train_seq[:1000]), "target ratings must be raw 1-5"
print("  [OK] Ratings are raw 1-5 (for embedding lookup in Model Training)")

# Check feature dimensions
feat = movie_features['Feature_Vector'].iloc[0]
assert len(feat) == MOVIE_FEAT_DIM, f"feature dim mismatch: {len(feat)} vs {MOVIE_FEAT_DIM}"
print(f"  [OK] Movie feature dim = {MOVIE_FEAT_DIM}")

# Check ID mappings
assert len(movie_id_to_idx) == NUM_MOVIES, "movie mapping size mismatch"
assert len(user_id_to_idx)  == NUM_USERS,  "user mapping size mismatch"
print(f"  [OK] ID mappings: {NUM_MOVIES} movies, {NUM_USERS} users")

# Check all sequence movie IDs exist in mapping
bad = [m for s in train_seq[:500]
         for m in s['seq_movie_ids']
         if m not in movie_id_to_idx]
assert len(bad) == 0, f"found {len(bad)} movie IDs not in movie_id_to_idx"
print("  [OK] All sequence movie IDs exist in movie_id_to_idx")

# Check counts
print(f"\n  Train : {len(train_seq):,}")
print(f"  Val   : {len(val_seq):,}")
print(f"  Test  : {len(test_sequences):,}")
print(f"  Total : {len(train_seq)+len(val_seq)+len(test_sequences):,}")

print("\n  All sanity checks passed!")

PHASE 1 SANITY CHECKS
  [OK] Sequence dict has all required keys
  [OK] All sequences have length 15
  [OK] Ratings are raw 1-5 (for embedding lookup in Model Training)
  [OK] Movie feature dim = 792
  [OK] ID mappings: 3706 movies, 6040 users
  [OK] All sequence movie IDs exist in movie_id_to_idx

  Train : 636,443
  Val   : 70,715
  Test  : 202,451
  Total : 909,609

  All sanity checks passed!
